# Extract Goals from Transcripts - Moving Window

This notebook proceeds to extract goals from interview transcripts in two stages: (1) generate goals from a moving window over the transcript; (2) trace generated goals back to the source text in the transcript.

In [1]:
from openai import OpenAI

client = OpenAI()
gpt4o_model = "gpt-4o-2024-08-06"
gpt5_2_model = "gpt-5.2-2025-12-11"

def prompt_model(prompt, model=gpt5_2_model):
    response = client.chat.completions.create(
      model=model,
      messages=[
        {
          "role": "system",
          "content": "You are a business analyst collecting requirements for a software application. Your job is to review interview transcripts between an interviewer who is a business analyst and a stakeholder who is a prospective user of the application. The interviewer will ask the stakeholder questions to identify their requirements for the application."
        },
        {
          "role": "user",
          "content": prompt
        }
      ],
      response_format= {"type": "json_object"}
    )
    #print(response)
    return response.choices[0].message.content

In [2]:
import json

# data_path = 'data1'
data_path = 'data1_gpt52'

data = json.load(open('%s/transcripts.json' % data_path, 'r'))
print('Read %i transcripts' % len(data['transcript']))

Read 34 transcripts


In [3]:
# Each goal should be written in the format of VERB[STATE] where VERB is one of Achieve, Maintain or Avoid, and STATE describes a system state to be achieved, maintained or avoided. 

prompt1 = """Read the following interview transcript excerpt and respond with any goals that the speaker expresses.
Write each goal in general language so that it describes only one action, and do not include references to applications,
products or services in the goal. Only write goals that can be traced to specific phrases in the speech.
Respond with the goals in a JSON list of strings.

%s

Goals:
"""

prompt2 = """Read the following goals in JSON format and identify the substrings in the interview transcript
excerpt from which the goals were generated. Respond in JSON using the format
{'goal': 'goal statement', 'phrases': ['phrase1', 'phrase2']}

%s

%s

Response: """

def extract_transcript_goals(transcript):
    extracted = []
    excerpt = '\n'.join(['%s: %s\n' % (t['speaker'], t['text']) for t in transcript])
    p1 = prompt1 % excerpt
    #print(p1)
    r1 = prompt_model(p1)
    #print(r1)

    goals = []
    phrases = []
    if r1:
        goals = json.loads(r1)

        p2 = prompt2 % (r1, excerpt)
        r2 = prompt_model(p2)
        #print(r2)
        if r2:
            phrases = json.loads(r2)

    return {'excerpt': excerpt, 'goals': goals, 'phrases': phrases} 


In [4]:
def extract_goals(data, i, results):
    print('Extracting goals from transcript %i with %i turns' % (i, len(data['transcript'][i])), end='')
    
    results[data['ids'][i]] = []
    for j in range(0, len(data['transcript'][i]), 2):
        print(' .', end='')
        
        e = extract_transcript_goals(data['transcript'][i][j:j+4])  # moving window
        results[data['ids'][i]].append(e)
    print(' .done!')

In [6]:
#results = {}
results = json.load(open('%s/extracted-moving.json' % data_path))

In [17]:
extract_goals(data, 0, results)

Extracting goals from transcript 0 with 84 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [21]:
extract_goals(data, 1, results)

Extracting goals from transcript 1 with 50 turns . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [22]:
extract_goals(data, 2, results)

Extracting goals from transcript 2 with 27 turns . . . . . . . . . . . . . . .done!


In [23]:
extract_goals(data, 3, results)

Extracting goals from transcript 3 with 191 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [24]:
extract_goals(data, 4, results)

Extracting goals from transcript 4 with 104 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [27]:
extract_goals(data, 5, results)

Extracting goals from transcript 5 with 70 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [28]:
extract_goals(data, 6, results)

Extracting goals from transcript 6 with 111 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [29]:
extract_goals(data, 7, results)

Extracting goals from transcript 7 with 59 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [30]:
extract_goals(data, 8, results)

Extracting goals from transcript 8 with 111 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [33]:
extract_goals(data, 9, results)

Extracting goals from transcript 9 with 86 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [34]:
extract_goals(data, 10, results)

Extracting goals from transcript 10 with 56 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [35]:
extract_goals(data, 11, results)

Extracting goals from transcript 11 with 118 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [36]:
extract_goals(data, 12, results)

Extracting goals from transcript 12 with 46 turns . . . . . . . . . . . . . . . . . . . . . . . .done!


In [37]:
extract_goals(data, 13, results)

Extracting goals from transcript 13 with 70 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [31]:
extract_goals(data, 14, results)

Extracting goals from transcript 14 with 114 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [7]:
extract_goals(data, 15, results)

Extracting goals from transcript 15 with 78 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [8]:
extract_goals(data, 16, results)

Extracting goals from transcript 16 with 65 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [9]:
extract_goals(data, 17, results)

Extracting goals from transcript 17 with 48 turns . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [10]:
extract_goals(data, 18, results)

Extracting goals from transcript 18 with 98 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [13]:
extract_goals(data, 19, results)

Extracting goals from transcript 19 with 77 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [14]:
extract_goals(data, 20, results)

Extracting goals from transcript 20 with 77 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [15]:
extract_goals(data, 21, results)

Extracting goals from transcript 21 with 33 turns . . . . . . . . . . . . . . . . . .done!


In [16]:
extract_goals(data, 22, results)

Extracting goals from transcript 22 with 46 turns . . . . . . . . . . . . . . . . . . . . . . . .done!


In [17]:
extract_goals(data, 23, results)

Extracting goals from transcript 23 with 36 turns . . . . . . . . . . . . . . . . . . .done!


In [18]:
extract_goals(data, 24, results)

Extracting goals from transcript 24 with 19 turns . . . . . . . . . . .done!


In [19]:
extract_goals(data, 25, results)

Extracting goals from transcript 25 with 67 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [20]:
extract_goals(data, 26, results)

Extracting goals from transcript 26 with 60 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [21]:
extract_goals(data, 27, results)

Extracting goals from transcript 27 with 163 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [22]:
extract_goals(data, 28, results)

Extracting goals from transcript 28 with 142 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [23]:
extract_goals(data, 29, results)

Extracting goals from transcript 29 with 50 turns . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [24]:
extract_goals(data, 30, results)

Extracting goals from transcript 30 with 116 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [25]:
extract_goals(data, 31, results)

Extracting goals from transcript 31 with 75 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [26]:
extract_goals(data, 32, results)

Extracting goals from transcript 32 with 118 turns . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . . .done!


In [27]:
extract_goals(data, 33, results)

Extracting goals from transcript 33 with 27 turns . . . . . . . . . . . . . . .done!


In [ ]:


#i = len(data['transcript'])-1
i = 0
while i < len(data['transcript']):
    
    i += 1

In [32]:
json.dump(results, open('%s/extracted-moving.json' % data_path, 'w')) # moved under .../extracted-moving.json

In [33]:
print(results.keys())

dict_keys(['32', '35', '34', '33', '20', '18', '27', '9', '11', '7', '29', '16', '6', '28', '17', '10', '19', '26', '8', '21', '36', '31', '30', '24', '23', '4', '15', '3', '12', '13', '5', '14', '22', '25'])


In [30]:
len(results)

34